# Luv2Code RAG Project - Q&A App using LangChain, OpenAI and Pinecone

### Requirements and Environment Setup

In [5]:
pip install -q -r ./requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# .env files are not included in the repository for security reasons (as specified
# in the .gitignore file), so create your own .env file with the following content:
# OPENAI_API_KEY=""
# PINECONE_API_KEY=""
# LANGCHAIN_API_KEY=""
# LANGCHAIN_TRACING_V2="false"

True

### Loading Documents

In [7]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    # if extension == '.pdf':
    #     from langchain_pymupdf4llm import PyMuPDF4LLMLoader
    #     os.environ['TESSDATA_PREFIX'] = "./tessdata"
    #     print(f'Loading {file} with OCR enabled via data path: {os.environ['TESSDATA_PREFIX']}')
    #     loader = PyMuPDF4LLMLoader(file)
    if extension == '.pdf':
        from langchain_classic.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain_classic.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain_classic.document_loaders import TextLoader
        loader = TextLoader(file)
        print(f'Loading {file}')
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data

### Chunking Data

In [8]:
def chunk_data(data, chunk_size=256):
    from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

### Calculating Cost

In [9]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

### Embedding and Uploading to the Pinecone Vector Database

In [10]:
# - give every chunk a deterministic Pinecone ID based on
# - its document source, page, page label and chunk text
# - check each chunk individually
# - skip chunks for which ID already exists in Pinecone
# - only insert new chunks

def insert_or_fetch_embeddings(index_name, chunks, namespace):
    """Insert only new embeddings into Pinecone, then return the vector store."""
    import hashlib
    import time
    import pinecone
    from langchain_community.vectorstores import Pinecone as PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    def make_chunk_id(chunk):
        source = str(chunk.metadata.get("source", "unknown-source")).replace("\\", "/")
        page = str(chunk.metadata.get("page", "unknown-page"))
        page_label = str(chunk.metadata.get("page_label", "unknown-page-label"))
        id_text = f"{source}|{page}|{page_label}|{chunk.page_content}"
        return hashlib.sha256(id_text.encode("utf-8")).hexdigest()

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"Creating index {index_name} ... ", end="")

        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)

        print("Ok")

    pinecone_index = pc.Index(index_name)

    vector_store = PineconeVectorStore(
        index=pinecone_index,
        embedding=embeddings,
        text_key="text",
        namespace=namespace
    )

    chunk_ids = [make_chunk_id(chunk) for chunk in chunks]
    chunks_to_add = []
    ids_to_add = []

    fetch_batch_size = 100

    for i in range(0, len(chunk_ids), fetch_batch_size):
        batch_ids = chunk_ids[i:i + fetch_batch_size]
        batch_chunks = chunks[i:i + fetch_batch_size]

        fetch_response = pinecone_index.fetch(
            ids=batch_ids,
            namespace=namespace
        )

        existing_vectors = getattr(fetch_response, "vectors", None)

        if existing_vectors is None:
            existing_vectors = fetch_response.get("vectors", {})

        existing_ids = set(existing_vectors.keys())

        for chunk_id, chunk in zip(batch_ids, batch_chunks):
            if chunk_id not in existing_ids:
                ids_to_add.append(chunk_id)
                chunks_to_add.append(chunk)

    if chunks_to_add:
        print(f"Adding {len(chunks_to_add)} new documents to index {index_name} ...\n", end="")

        embedding_batch_size = 100
        total_batches = (len(chunks_to_add) + embedding_batch_size - 1) // embedding_batch_size

        for i in range(0, len(chunks_to_add), embedding_batch_size):
            batch_chunks = chunks_to_add[i:i + embedding_batch_size]
            batch_ids = ids_to_add[i:i + embedding_batch_size]
            batch_texts = [chunk.page_content for chunk in batch_chunks]

            batch_number = (i // embedding_batch_size) + 1
            print(f"Adding batch {batch_number}/{total_batches} ({len(batch_chunks)} documents) ... ", end="")

            batch_embeddings = embeddings.embed_documents(batch_texts)

            vectors = []

            for chunk_id, chunk, embedding in zip(batch_ids, batch_chunks, batch_embeddings):
                metadata = dict(chunk.metadata)
                metadata["text"] = chunk.page_content

                vectors.append({
                    "id": chunk_id,
                    "values": embedding,
                    "metadata": metadata
                })

            pinecone_index.upsert(
                vectors=vectors,
                namespace=namespace
            )

            print("Ok")

        print("All batches added successfully!")

        # Give Pinecone a moment to make freshly upserted vectors visible to fetch/stats.
        time.sleep(5)
    else:
        print(f"All {len(chunks)} documents already exist in index {index_name}. No new documents added.")

    return vector_store

In [11]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('OK')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('OK')

### Asking and Getting Answers

In [12]:
def ask_and_get_answer(vector_store, q, k=3):
    from langchain_classic.chains import RetrievalQA
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=1)

    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    chain = RetrievalQA.from_chain_type(llm=llm,
                                        chain_type="stuff",
                                        retriever=retriever,
                                        return_source_documents=True)
    
    answer = chain.invoke(q)
    return answer

### Running Code

#### Load Documents and Chunk Data

In [13]:
from pathlib import Path

# amend accordingly
folder_paths = [
    Path("../Luv2Code_Docs"),
]

data = []

for folder_path in folder_paths:
    for file_path in folder_path.iterdir():
        if file_path.suffix.lower() in [".pdf", ".docx", ".txt"]:
            document_pages = load_document(str(file_path))

            if document_pages is not None:
                for page in document_pages:
                    page.metadata["page_label"] = page.metadata.get("page_label", "Unknown page")
                    page.metadata["page"] = page.metadata.get("page", "Unknown page")

                data.extend(document_pages)

print(f"There are {len(data)} pages in total")

if data:
    print(data[0].page_content)
    print(data[0].metadata)
    print(f"There are {len(data[0].page_content)} characters on the first page")

Loading ..\Luv2Code_Docs\FSAS_01-introduction-and-setup.pdf
Loading ..\Luv2Code_Docs\FSAS_02-typescript-crash-course.pdf
Loading ..\Luv2Code_Docs\FSAS_03-angular-crash-course.pdf
Loading ..\Luv2Code_Docs\FSAS_04-product-list-shopping-cart-checkout.pdf
Loading ..\Luv2Code_Docs\FSAS_05-security.pdf
Loading ..\Luv2Code_Docs\FSAS_06-credit-card-payment-processing-with-stripe.pdf
Loading ..\Luv2Code_Docs\FSAS_07-course-summary.pdf
Loading ..\Luv2Code_Docs\JPLA_01-java-playwright-master-web-test-automation.pdf
Loading ..\Luv2Code_Docs\MJDP_01-master-java-design-patterns.pdf
Loading ..\Luv2Code_Docs\SBRA_01-introduction.pdf
Loading ..\Luv2Code_Docs\SBRA_02-spring-boot-rest-project-1.pdf
Loading ..\Luv2Code_Docs\SBRA_03-spring-boot-rest-project-2.pdf


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 165 0 (offset 0)
Ignoring wrong pointing object 212 0 (offset 0)
Ignoring wrong pointing object 230 0 (offset 0)
Ignoring wrong pointing object 275 0 (offset 0)
Ignoring wrong pointing object 283 0 (offset 0)
Ignoring wrong pointing object 319 0 (offset 0)
Ignoring wrong pointing object 341 0 (offset 0)
Ignoring wrong pointing object 387 0 (offset 0)
Ignoring wrong pointing object 413 0 (offset 0)
Ignoring wrong pointing object 449 0 (offset 0)
Ignoring wrong pointing object 454 0 (offset 0)


Loading ..\Luv2Code_Docs\SBRA_04-spring-boot-rest-project-3-part-1.pdf
Loading ..\Luv2Code_Docs\SBRA_05-spring-boot-rest-project-3-part-2.pdf


Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 115 0 (offset 0)
Ignoring wrong pointing object 152 0 (offset 0)
Ignoring wrong pointing object 198 0 (offset 0)
Ignoring wrong pointing object 247 0 (offset 0)
Ignoring wrong pointing object 255 0 (offset 0)
Ignoring wrong pointing object 299 0 (offset 0)
Ignoring wrong pointing object 343 0 (offset 0)
Ignoring wrong pointing object 357 0 (offset 0)
Ignoring wrong pointing object 415 0 (offset 0)
Ignoring wrong pointing object 440 0 (offset 0)
Ignoring wrong pointing object 468 0 (offset 0)
Ignoring wrong pointing object 511 0 (offset 0)
Ignoring wrong pointing object 548 0 (offset 0)
Ignoring wrong pointing object 614 0 (offset 0)
Ignoring wrong pointing object 642 0 (offset 0)
Ignoring wrong pointing object 717 0 (offset 0)
Ignoring wrong pointing object 736 0 (offset 0)
Ignoring wrong pointing object 763 0 (offset 0)
Ignoring wrong pointing object 775 0 (offset 0)
Ignoring wrong pointing object 809 0 (off

Loading ..\Luv2Code_Docs\SBRA_06-spring-boot-rest-project-4.pdf
Loading ..\Luv2Code_Docs\SBRA_07-summary.pdf
Loading ..\Luv2Code_Docs\SBUT_01-spring-boot-unit-testing-introduction.pdf
Loading ..\Luv2Code_Docs\SBUT_02-junit-review.pdf
Loading ..\Luv2Code_Docs\SBUT_03-test-driven-development.pdf
Loading ..\Luv2Code_Docs\SBUT_04-spring-boot-unit-testing-support.pdf
Loading ..\Luv2Code_Docs\SBUT_05-mocking-with-mockito.pdf
Loading ..\Luv2Code_Docs\SBUT_06-reflection-utils.pdf
Loading ..\Luv2Code_Docs\SBUT_07-database-integration-testing.pdf


Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 111 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 139 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)
Ignoring wrong pointing object 78 0 (offset 0)


Loading ..\Luv2Code_Docs\SBUT_08-testing-spring-boot-mvc-web-apps.pdf
Loading ..\Luv2Code_Docs\SBUT_09-testing-spring-boot-mvc-web-apps-student-grades.pdf
Loading ..\Luv2Code_Docs\SBUT_10-testing-spring-boot-mvc-web-apps-set-up-sql-scripts.pdf
Loading ..\Luv2Code_Docs\SBUT_11-testing-spring-boot-mvc-web-apps-student-information-and-grades.pdf
Loading ..\Luv2Code_Docs\SBUT_12-testing-spring-boot-rest-apis.pdf
Loading ..\Luv2Code_Docs\SBUT_13-course-summary.pdf
There are 2245 pages in total
Angular and Spring Boot
{'producer': 'PDFelement Pro', 'creator': 'luv2code', 'creationdate': '2024-02-05 12:54:16 +0000', 'title': '01-introduction-and-setup.pdf', 'author': 'www.luv2code.com', 'moddate': '2024-02-05T07:54:17-05:00', 'keywords': 'luv2code', 'subject': 'luv2code', 'source': '..\\Luv2Code_Docs\\FSAS_01-introduction-and-setup.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
There are 23 characters on the first page


In [14]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[len(chunks) - 1].page_content)

4243
www.luv2code.com
Contact Us
Eric Roby 
 
Coding with Roby 
 
www.ericjroby.com
Chád Darby 
www.luv2code.com


In [15]:
print_embedding_cost(chunks)

Total Tokens: 178682
Embedding Cost in USD: 0.003574


In [16]:
import pinecone
pc = pinecone.Pinecone()
pc.list_indexes()

IndexList([<name='anthony-burgess-rag', dim=1536, ready=True>, <name='luv2code-rag', dim=1536, ready=True>, <name='ou-comp-it-degree', dim=1536, ready=True>])

In [17]:
from pinecone import ServerlessSpec
index_name = 'luv2code-rag'
if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name} ...', end='')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print('Index created!')
else:
    print(f'Index {index_name} already exists!')
# pc.list_indexes()[0]['name']
pc.describe_index('luv2code-rag')

Index luv2code-rag already exists!


IndexModel(name='luv2code-rag', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://luv2code-rag-moalxso.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [25]:
index_name = 'luv2code-rag'
index = pc.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=4243, metric='cosine', namespaces=1)

In [19]:
# Only use this if you need to
# delete_pinecone_index('luv2code-rag')

In [20]:
index_name = 'luv2code-rag'
namespace = 'luv2code-rag'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks, namespace=namespace)

C:\Users\james\AppData\Local\Temp\ipykernel_23196\3757734243.py:48: LangChainDeprecationWarning: The class `Pinecone` was deprecated in LangChain 0.0.18 and will be removed in 1.0. An updated version of the class exists in the `langchain-pinecone package and should be used instead. To use it run `pip install -U `langchain-pinecone` and import as `from `langchain_pinecone import Pinecone``.
  vector_store = PineconeVectorStore(


Adding 4243 new documents to index luv2code-rag ...
Adding batch 1/43 (100 documents) ... Ok
Adding batch 2/43 (100 documents) ... Ok
Adding batch 3/43 (100 documents) ... Ok
Adding batch 4/43 (100 documents) ... Ok
Adding batch 5/43 (100 documents) ... Ok
Adding batch 6/43 (100 documents) ... Ok
Adding batch 7/43 (100 documents) ... Ok
Adding batch 8/43 (100 documents) ... Ok
Adding batch 9/43 (100 documents) ... Ok
Adding batch 10/43 (100 documents) ... Ok
Adding batch 11/43 (100 documents) ... Ok
Adding batch 12/43 (100 documents) ... Ok
Adding batch 13/43 (100 documents) ... Ok
Adding batch 14/43 (100 documents) ... Ok
Adding batch 15/43 (100 documents) ... Ok
Adding batch 16/43 (100 documents) ... Ok
Adding batch 17/43 (100 documents) ... Ok
Adding batch 18/43 (100 documents) ... Ok
Adding batch 19/43 (100 documents) ... Ok
Adding batch 20/43 (100 documents) ... Ok
Adding batch 21/43 (100 documents) ... Ok
Adding batch 22/43 (100 documents) ... Ok
Adding batch 23/43 (100 documents

In [22]:
q = 'Write four paragraphs about Spring Boot unit testing. Answer from the provided context only.'
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])
print("\nReference(s):")

for doc in answer["source_documents"]:
    source = doc.metadata.get("source", "Unknown document")
    page_label = doc.metadata.get("page_label")
    page = doc.metadata.get("page")

    print(f"\nDocument source: {source}")
    print(f"Page label: {page_label}")
    print(f"Page index: {page}")

Spring Boot unit testing is an essential aspect of ensuring the reliability and functionality of applications developed using the Spring framework. One key requirement for effective unit testing in Spring Boot is access to the Spring Application Context. This context acts as a central registry for the beans, allowing the test to utilize the configurations and components defined within the application. By leveraging the Application Context, developers can create test scenarios that closely mimic the actual running application.

Another important element in Spring Boot unit testing is support for Spring dependency injection. Dependency injection simplifies the process of managing dependencies between various components in an application. During unit tests, developers can easily simulate different scenarios by injecting mock or stubbed versions of dependencies into the tests. This capability allows for isolated testing of individual components, ensuring that each part of the application f

#### While Loop for Asking Questions

In [24]:
import time
i = 1
print('Enter your questions (or write Quit to quit).')

while True:
    q = input(f'Question #{i}: ') + ' Answer from the provided context only.'
    i = i + 1
    if q.lower().startswith('quit'):
        print('\nQuitting ... bye bye!')
        time.sleep(2)
        break
    
    answer = ask_and_get_answer(vector_store, q)
    print(f'\n{"-" * 50}')
    print(f'\nQuery: ' + answer['query'])
    print(f'\nAnswer: ' + answer['result'])
    print("\nReference(s):")

    for doc in answer["source_documents"]:
        source = doc.metadata.get("source", "Unknown document")
        page_label = doc.metadata.get("page_label")
        page = doc.metadata.get("page")

        print(f"\nDocument: {source}")
        print(f"Page label: {page_label}")
        print(f"Page index: {page}")


Enter your questions (or write Quit to quit).

--------------------------------------------------

Query: Describe the factory desgin pattern in four paragraphs. Answer from the provided context only.

Answer: The Factory Design Pattern provides a way to create objects using a common interface without exposing the creation logic. This design pattern is particularly useful when the type of object that needs to be created may vary, as it allows for a more flexible and decoupled approach to object creation. By using this pattern, developers can avoid the complexities of instantiating classes directly and instead delegate the responsibility of object creation to a factory.

One of the main benefits of the Factory Design Pattern is that it enables the actual object type to be decided at runtime. This is accomplished by using subclasses or factory methods that can return different types of objects based on input parameters or conditions. This flexibility allows for scalable and adaptable cod